# RAG Multimodal avec un Agent IA (ReAct & LangGraph)

**Objectifs :**
- extraire le texte et les images d’un document PDF ;
- calculer des `embeddings` multimodaux (texte et image dans un même espace vectoriel) ;
- indexer ces embeddings dans une base vectorielle (`ChromaDB`) avec leurs métadonnées ;
- implémenter une recherche par similarité cosinus à partir d’un texte **ou** d’une image ;
- transformer le `Retriever` en outil (`Tool`) utilisable par un LLM ;
- construire un agent basé sur le pattern **ReAct** avec `LangGraph` ;
- tester l’agent sur des requêtes multimodales.

## 0. Préparation

Le notebook fonctionne sur CPU. S’il existe un GPU CUDA, PyTorch pourra l’utiliser automatiquement.

Décommentez la cellule suivante pour installer les dépendances (une seule fois).

In [ ]:
# !pip install pymupdf pillow matplotlib numpy sentence-transformers chromadb
# !pip install langchain-core langgraph langchain-ollama
# (ou langchain-openai / langchain-anthropic / langchain-groq selon le LLM choisi à l'étape 6)

In [ ]:
import os
import io
import re
import hashlib
from pathlib import Path

import fitz  # PyMuPDF
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Device utilisé:", device)

## 1. Le workflow général du RAG multimodal

Le cycle complet est :

**PDF → Extraction (texte + images) → Embeddings → Base vectorielle → Retrieval → Outil → Agent → Réponse**

Le principe du **RAG** (*Retrieval-Augmented Generation*) : au lieu de laisser le LLM répondre seul, on lui fournit d’abord des passages pertinents retrouvés dans nos documents.

Le côté **multimodal** vient du fait que le texte *et* les images sont projetés dans le **même espace vectoriel** grâce à un modèle de type `CLIP`.

## 2. Étape 1 — Chargement du document et extraction du contenu

Nous utilisons `PyMuPDF` (`fitz`) pour lire le PDF page par page :
- le **texte** est découpé en petits morceaux (*chunks*) ;
- les **images** sont extraites puis sauvegardées sur le disque.

> Le modèle `CLIP` ne lit que **77 tokens** de texte au maximum. Il faut donc des chunks courts (~300 caractères).

Placez votre PDF dans le dossier du notebook et modifiez `PDF_PATH`. Si le fichier n’existe pas, la cellule suivante génère un **PDF de démonstration** contenant du texte et deux graphiques.

In [ ]:
PDF_PATH = "document.pdf"          # <-- remplacez par votre propre PDF
IMG_DIR = Path("extracted_images")
IMG_DIR.mkdir(exist_ok=True)

In [ ]:
def create_demo_pdf(path):
    # Crée un petit PDF (texte + 2 graphiques) pour tester le pipeline
    years = [2019, 2020, 2021, 2022, 2023]

    # Graphique 1 : diagramme en barres
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(years, [12, 18, 25, 33, 41], color="tab:blue")
    ax.set_title("Production d'énergie solaire (TWh)")
    ax.set_xlabel("Année"); ax.set_ylabel("TWh")
    fig.tight_layout(); fig.savefig("demo_bar.png", dpi=120); plt.close(fig)

    # Graphique 2 : camembert
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.pie([45, 30, 15, 10], labels=["Solaire", "Éolien", "Hydraulique", "Autres"], autopct="%1.0f%%")
    ax.set_title("Répartition du mix énergétique")
    fig.tight_layout(); fig.savefig("demo_pie.png", dpi=120); plt.close(fig)

    doc = fitz.open()

    page = doc.new_page()
    page.insert_textbox(fitz.Rect(50, 50, 545, 400), (
        "Les énergies renouvelables en forte croissance.\n\n"
        "La production d'énergie solaire a plus que triplé entre 2019 et 2023. "
        "Cette progression s'explique par la baisse du coût des panneaux photovoltaïques. "
        "Les investissements dans les parcs solaires ont fortement augmenté dans les pays ensoleillés."
    ), fontsize=12)
    page.insert_image(fitz.Rect(100, 300, 460, 520), filename="demo_bar.png")

    page = doc.new_page()
    page.insert_textbox(fitz.Rect(50, 50, 545, 400), (
        "Le mix énergétique est de plus en plus diversifié.\n\n"
        "Le solaire représente désormais près de la moitié de la production renouvelable, "
        "devant l'éolien et l'hydraulique. "
        "Le stockage par batteries reste le principal défi pour stabiliser le réseau électrique."
    ), fontsize=12)
    page.insert_image(fitz.Rect(150, 300, 450, 600), filename="demo_pie.png")

    doc.save(path)
    doc.close()

if not os.path.exists(PDF_PATH):
    create_demo_pdf(PDF_PATH)
    print("PDF de démonstration créé :", PDF_PATH)
else:
    print("PDF trouvé :", PDF_PATH)

In [ ]:
def split_text(text, max_chars=300):
    # Découpe un texte en chunks courts, en respectant les phrases
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) + 1 <= max_chars:
            current = f"{current} {s}".strip()
        else:
            if current:
                chunks.append(current)
            current = s[:max_chars]
    if current:
        chunks.append(current)
    return chunks


def extract_pdf_content(pdf_path, min_size=100, max_chars=300):
    # Retourne une liste d'éléments : {"type", "page", "content"}
    doc = fitz.open(pdf_path)
    items = []
    for page_num, page in enumerate(doc, start=1):
        # --- Texte ---
        for chunk in split_text(page.get_text("text"), max_chars):
            items.append({"type": "text", "page": page_num, "content": chunk})

        # --- Images ---
        for img_idx, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base = doc.extract_image(xref)
            image = Image.open(io.BytesIO(base["image"])).convert("RGB")
            if min(image.size) < min_size:      # on ignore les icônes / logos minuscules
                continue
            img_path = IMG_DIR / f"{Path(pdf_path).stem}_p{page_num}_img{img_idx}.png"
            image.save(img_path)
            items.append({"type": "image", "page": page_num, "content": str(img_path)})
    doc.close()
    return items

items = extract_pdf_content(PDF_PATH)

n_text = sum(i["type"] == "text" for i in items)
n_img = sum(i["type"] == "image" for i in items)
print(f"{len(items)} éléments extraits : {n_text} textes, {n_img} images")

In [ ]:
for it in items[:6]:
    print(f"[page {it['page']}] ({it['type']}) {it['content'][:100]}")

In [ ]:
image_items = [i for i in items if i["type"] == "image"]

fig, axes = plt.subplots(1, max(len(image_items), 2), figsize=(10, 4))
for ax, it in zip(axes, image_items):
    ax.imshow(Image.open(it["content"]))
    ax.set_title(f"Page {it['page']}")
    ax.axis("off")
plt.show()

## 3. Les embeddings multimodaux avec CLIP

`CLIP` apprend à placer un texte et une image **proches l’un de l’autre** dans l’espace vectoriel lorsqu’ils parlent de la même chose.

Nous utilisons le modèle `clip-ViT-B-32` via `sentence-transformers`. Les vecteurs sont **normalisés** (norme = 1), ce qui permet de calculer la similarité cosinus par un simple produit scalaire :

\[
\cos(u, v) = \frac{u \cdot v}{\|u\|\,\|v\|} = u \cdot v \quad \text{si } \|u\| = \|v\| = 1
\]

In [ ]:
from sentence_transformers import SentenceTransformer

clip_model = SentenceTransformer("clip-ViT-B-32", device=device)

def embed_texts(texts):
    return clip_model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)

def embed_images(images):
    # images : liste d'objets PIL.Image
    return clip_model.encode(images, normalize_embeddings=True, convert_to_numpy=True)

e = embed_texts(["un graphique en barres"])
print("Shape embedding texte :", e.shape)
print("Norme :", np.linalg.norm(e[0]))

### Vérification : texte ↔ image

Si l’espace est bien partagé, la phrase *« un diagramme circulaire »* doit être plus proche du camembert que du diagramme en barres.

In [ ]:
imgs = [Image.open(i["content"]) for i in image_items]
img_emb = embed_images(imgs)

queries = ["un diagramme en barres", "un diagramme circulaire (camembert)"]
txt_emb = embed_texts(queries)

sim = txt_emb @ img_emb.T      # similarité cosinus (vecteurs normalisés)
for q, row in zip(queries, sim):
    print(q, "->", np.round(row, 3))

## 4. Étape 3 — Base vectorielle et indexation

Nous utilisons `ChromaDB` avec la **distance cosinus**.

Pour gérer les **réindexations**, chaque élément reçoit un identifiant calculé par hachage `SHA-256` de son contenu :
- même contenu → même identifiant → `upsert` met à jour l’entrée au lieu de la dupliquer ;
- contenu modifié → nouvel identifiant → nouvelle entrée.

Chaque vecteur est stocké avec ses **métadonnées** : type de contenu, numéro de page, chemin de l’image.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chroma_db")

collection = chroma_client.get_or_create_collection(
    name="rag_multimodal",
    metadata={"hnsw:space": "cosine"}      # distance cosinus
)

print("Collection :", collection.name)
print("Nombre d'éléments déjà indexés :", collection.count())

In [ ]:
def compute_doc_id(item):
    # Identifiant SHA-256 : type + page + contenu (texte ou octets de l'image)
    h = hashlib.sha256()
    h.update(item["type"].encode())
    h.update(str(item["page"]).encode())
    if item["type"] == "image":
        h.update(Path(item["content"]).read_bytes())
    else:
        h.update(item["content"].encode("utf-8"))
    return h.hexdigest()


def index_items(items, batch_size=32):
    ids, embeddings, metadatas, documents = [], [], [], []

    text_items = [i for i in items if i["type"] == "text"]
    image_items_ = [i for i in items if i["type"] == "image"]

    # Embeddings des textes
    if text_items:
        emb = embed_texts([i["content"] for i in text_items])
        for it, e in zip(text_items, emb):
            ids.append(compute_doc_id(it))
            embeddings.append(e.tolist())
            documents.append(it["content"])
            metadatas.append({"type": "text", "page": it["page"], "image_path": ""})

    # Embeddings des images
    if image_items_:
        emb = embed_images([Image.open(i["content"]) for i in image_items_])
        for it, e in zip(image_items_, emb):
            ids.append(compute_doc_id(it))
            embeddings.append(e.tolist())
            documents.append(f"[image page {it['page']}]")
            metadatas.append({"type": "image", "page": it["page"], "image_path": it["content"]})

    # upsert = insertion ou mise à jour (pas de doublons)
    for start in range(0, len(ids), batch_size):
        sl = slice(start, start + batch_size)
        collection.upsert(
            ids=ids[sl], embeddings=embeddings[sl],
            metadatas=metadatas[sl], documents=documents[sl]
        )
    return len(ids)

n = index_items(items)
print(f"{n} éléments traités -> collection.count() = {collection.count()}")

### Test de réindexation

Si on relance l’indexation, le nombre d’éléments ne doit **pas** changer.

In [ ]:
index_items(items)
print("Après réindexation :", collection.count())

## 5. Étape 4 — Recherche multimodale (Retrieval)

La fonction `retrieve` accepte comme requête :
- un **texte** (`str`) ;
- une **image** (objet `PIL.Image` ou chemin vers un fichier image).

La requête est encodée avec CLIP, puis comparée aux vecteurs de la base. ChromaDB retourne une *distance cosinus* ; on la convertit en score de similarité :

\[
\text{score} = 1 - d_{\cos}
\]

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def is_image_query(query):
    if isinstance(query, Image.Image):
        return True
    return isinstance(query, (str, Path)) and Path(str(query)).suffix.lower() in IMG_EXT and os.path.exists(str(query))


def retrieve(query, k=4, content_type=None):
    # query : texte, chemin d'image ou PIL.Image
    if is_image_query(query):
        img = query if isinstance(query, Image.Image) else Image.open(query)
        q_emb = embed_images([img])[0]
    else:
        q_emb = embed_texts([str(query)])[0]

    kwargs = {"query_embeddings": [q_emb.tolist()], "n_results": k}
    if content_type in ("text", "image"):
        kwargs["where"] = {"type": content_type}

    res = collection.query(**kwargs)

    results = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        results.append({
            "type": meta["type"],
            "page": meta["page"],
            "content": doc,
            "image_path": meta["image_path"],
            "score": 1 - dist,
        })
    return results


def show_results(results, title=""):
    n_img = sum(r["type"] == "image" for r in results)
    for r in results:
        if r["type"] == "text":
            print(f"[TEXTE | page {r['page']} | score {r['score']:.3f}] {r['content']}")
        else:
            print(f"[IMAGE | page {r['page']} | score {r['score']:.3f}] {r['image_path']}")
    if n_img:
        fig, axes = plt.subplots(1, n_img, figsize=(4 * n_img, 3.5))
        axes = np.atleast_1d(axes)
        for ax, r in zip(axes, [r for r in results if r["type"] == "image"]):
            ax.imshow(Image.open(r["image_path"]))
            ax.set_title(f"page {r['page']} — {r['score']:.3f}")
            ax.axis("off")
        plt.suptitle(title)
        plt.show()

### Requête textuelle

In [ ]:
res = retrieve("évolution de la production d'énergie solaire", k=4)
show_results(res, title="Requête texte")

### Requête image

On utilise une image extraite du PDF comme requête : le résultat le plus proche doit être elle-même.

In [ ]:
query_img_path = image_items[0]["content"]
res = retrieve(query_img_path, k=3)
show_results(res, title="Requête image")

### Filtrer par type de contenu

In [ ]:
print("--- Images uniquement ---")
show_results(retrieve("répartition en pourcentage", k=2, content_type="image"))

print("\n--- Textes uniquement ---")
show_results(retrieve("répartition en pourcentage", k=2, content_type="text"))

## 6. Étape 5 — Transformer le retriever en outil (`Tool`)

Un LLM ne peut pas appeler directement une fonction Python. On l’encapsule dans un **outil** qui expose :
- un **nom** ;
- une **description** (le LLM la lit pour décider quand utiliser l’outil) ;
- un **schéma d’arguments** (types déduits de la signature).

> La **docstring** est essentielle : c’est elle qui guide le modèle.

In [ ]:
from langchain_core.tools import tool

@tool
def retriever_tool(query: str, image_path: str = "", k: int = 4, content_type: str = "") -> str:
    """Recherche des informations dans le document PDF indexé (textes ET images).

    À utiliser dès qu'une question porte sur le contenu du document : chiffres, explications,
    graphiques, figures ou illustrations.

    Arguments :
    - query : question ou mots-clés décrivant ce qu'on cherche (en français).
    - image_path : (optionnel) chemin d'une image locale à utiliser comme requête
      pour retrouver des contenus visuellement ou sémantiquement proches.
    - k : nombre de résultats à retourner (défaut 4).
    - content_type : (optionnel) "text" pour ne chercher que du texte,
      "image" pour ne chercher que des images, vide pour les deux.

    Retourne une liste de passages avec leur type, leur page et leur score de similarité.
    Pour une image, retourne son chemin de fichier.
    """
    search_input = image_path if image_path and is_image_query(image_path) else query
    results = retrieve(search_input, k=k, content_type=content_type or None)

    if not results:
        return "Aucun résultat trouvé."

    lines = []
    for i, r in enumerate(results, start=1):
        if r["type"] == "text":
            lines.append(f"{i}. [TEXTE | page {r['page']} | score {r['score']:.2f}] {r['content']}")
        else:
            lines.append(f"{i}. [IMAGE | page {r['page']} | score {r['score']:.2f}] fichier : {r['image_path']}")
    return "\n".join(lines)

print("Nom        :", retriever_tool.name)
print("Arguments  :", retriever_tool.args)
print("Description:", retriever_tool.description[:200], "...")

### Test direct de l’outil (sans LLM)

In [ ]:
print(retriever_tool.invoke({"query": "stockage par batteries", "k": 2}))

## 7. Étape 6 — Construction de l’agent IA (ReAct & LangGraph)

### Architecture : MIND + BODY

- **MIND (cerveau)** : le LLM, qui raisonne et décide quoi faire ;
- **BODY (corps)** : les outils (ici `retriever_tool`), qui agissent sur le monde extérieur.

### Le pattern ReAct

**Reasoning → Action → Observation → (boucle) → Réponse finale**

1. le LLM **raisonne** sur la question ;
2. il décide d’**appeler** l’outil (ou de répondre directement) ;
3. le résultat de l’outil est une **observation** renvoyée au LLM ;
4. le LLM reprend la main, et recommence ou produit la réponse finale.

Avec `LangGraph`, cette boucle est un **graphe** à deux nœuds : `agent` (le LLM) et `tools` (l’exécution de l’outil), avec une **arête conditionnelle** qui décide si on continue ou si on s’arrête.

### Choix du LLM

Le LLM doit supporter le **tool calling**. Choisissez le fournisseur via la variable `LLM_PROVIDER`.

In [ ]:
import getpass

LLM_PROVIDER = "ollama"     # "ollama" | "openai" | "anthropic" | "groq"

def get_llm(provider):
    if provider == "ollama":                     # local et gratuit : `ollama pull llama3.1`
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3.1", temperature=0)
    if provider == "openai":
        if "OPENAI_API_KEY" not in os.environ:
            os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY : ")
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    if provider == "anthropic":
        if "ANTHROPIC_API_KEY" not in os.environ:
            os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY : ")
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
    if provider == "groq":
        if "GROQ_API_KEY" not in os.environ:
            os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY : ")
        from langchain_groq import ChatGroq
        return ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    raise ValueError("Fournisseur inconnu")

llm = get_llm(LLM_PROVIDER)
tools = [retriever_tool]

# On "branche" le corps (outils) sur le cerveau (LLM)
llm_with_tools = llm.bind_tools(tools)
print("LLM prêt :", type(llm).__name__)

### Définition du graphe

- le nœud `agent` appelle le LLM avec l’historique des messages ;
- le nœud `tools` exécute les appels d’outils demandés par le LLM ;
- `tools_condition` route vers `tools` si le LLM a demandé un outil, sinon vers `END`.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = (
    "Tu es un assistant qui répond aux questions sur un document PDF contenant du texte et des images. "
    "Utilise TOUJOURS l'outil retriever_tool avant de répondre à une question sur le document. "
    "Tu peux l'appeler plusieurs fois (par exemple une fois pour le texte, une fois pour les images). "
    "Appuie ta réponse uniquement sur les résultats obtenus, cite la page et, pour une image, son chemin de fichier. "
    "Si l'information n'est pas dans les résultats, dis-le clairement."
)

# --- MIND : le LLM raisonne et décide ---
def agent_node(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# --- BODY : exécution des outils ---
tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)   # -> "tools" ou END
builder.add_edge("tools", "agent")                        # Observation -> nouveau raisonnement

rag_agent = builder.compile()
print("Agent compilé.")

In [ ]:
# Visualisation du graphe (texte Mermaid)
print(rag_agent.get_graph().draw_mermaid())

> **Remarque :** `langgraph.prebuilt.create_react_agent(llm, tools)` construit exactement ce graphe en une seule ligne. Nous l’avons écrit à la main pour bien voir la boucle ReAct.

## 8. Étape 7 — Tests et validation de l’agent RAG

Nous définissons une fonction `ask` qui affiche toutes les étapes de la boucle ReAct :
- les **appels d’outil** décidés par le LLM (Action) ;
- les **résultats** renvoyés (Observation) ;
- la **réponse finale**.

In [ ]:
def ask(question, verbose=True):
    result = rag_agent.invoke({"messages": [HumanMessage(content=question)]})
    messages = result["messages"]

    n_calls = 0
    if verbose:
        for m in messages:
            if m.type == "human":
                print("QUESTION :", m.content, "\n")
            elif m.type == "ai" and getattr(m, "tool_calls", None):
                for tc in m.tool_calls:
                    n_calls += 1
                    print(f"ACTION      -> {tc['name']}({tc['args']})")
            elif m.type == "tool":
                print(f"OBSERVATION -> {m.content[:300]}{'...' if len(m.content) > 300 else ''}\n")
    else:
        n_calls = sum(len(getattr(m, 'tool_calls', []) or []) for m in messages)

    final = messages[-1].content
    print("\nRÉPONSE FINALE :\n", final)
    print(f"\n(Nombre d'appels à l'outil : {n_calls})")
    return result

### Test 1 — Question textuelle

In [ ]:
_ = ask("Comment a évolué la production d'énergie solaire entre 2019 et 2023, et pourquoi ?")

### Test 2 — Question sur une image

L’agent doit utiliser l’outil pour retrouver une **figure** (diagramme circulaire) et donner son chemin.

In [ ]:
_ = ask("Montre-moi la figure qui illustre la répartition du mix énergétique.")

### Test 3 — Requête hybride (texte + image)

Cette question nécessite de croiser **le texte** et **les graphiques** : l’agent devrait appeler l’outil plusieurs fois.

In [ ]:
_ = ask("Quel est le principal défi du stockage de l'énergie, et existe-t-il un graphique sur la part du solaire dans le mix ?")

### Test 4 — Requête avec une image en entrée

On fournit le chemin d’une image locale ; l’agent peut la passer à l’outil via `image_path`.

In [ ]:
img_path = image_items[0]["content"]
_ = ask(f"Trouve dans le document les contenus proches de cette image : {img_path}")

In [ ]:
# Question hors sujet : l'agent doit répondre honnêtement
_ = ask("Quelle est la capitale du Japon ?")

## 9. Analyse du comportement de l’agent

Pour chaque test, observez :

1. **Décision d’appel de l’outil** : l’agent appelle-t-il `retriever_tool` ? Avec quelle requête reformulée ?
2. **Choix des arguments** : utilise-t-il `content_type="image"` quand on cherche une figure ?
3. **Contexte récupéré** : les passages et images retournés sont-ils pertinents (scores élevés) ?
4. **Réponse finale** : est-elle fidèle aux résultats (pas d’invention), avec pages et chemins cités ?
5. **Questions hors sujet** : l’agent évite-t-il d’halluciner ?

### Pistes d’amélioration
- augmenter `k` ou ajouter un seuil de score minimal ;
- utiliser un LLM **vision** pour décrire les images récupérées ;
- remplacer `ChromaDB` par `FAISS` (`IndexFlatIP` sur des vecteurs normalisés = similarité cosinus) ;
- ajouter un **reranker** pour réordonner les résultats.